# Hijaiyah Sign Language Deep Learning Architecture (LPMQ Standard)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/1EqHWbovCn-lnD4LRET4GSBMm_w4u4ch4)

**Research Project**: Performance Comparison of Bi-LSTM, Hybrid LSTM-GRU, and Transformer Encoder on Recognizing Hijaiyah Letters with LPMQ Standard Signature  
**Author**: Zeiniah (UIN Syarif Hidayatullah Jakarta)  
**Collaborator**: Lajnah Pentashihan Mushaf Al-Qur'an (LPMQ) Kementerian Agama RI  
**Best Performance**: Transformer Encoder achieves **97.54% Accuracy** and **97.51% F1-Score**.

In [ ]:
# @title 1. Mount Google Drive & Environment Setup
from google.colab import drive
drive.mount('/content/drive')

!pip install mediapipe opencv-python scikit-learn seaborn matplotlib --quiet
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mediapipe as mp
import tensorflow as tf
print(f"TensorFlow Version: {tf.__version__}")

In [ ]:
# @title 2. MediaPipe Holistic 75-Keypoints Feature Extraction Function
mp_holistic = mp.solutions.holistic

def extract_keypoints(results):
    row = []
    # Right Hand Landmarks (21 points x 3 coordinates = 63 features)
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            row.extend([lm.x, lm.y, lm.z])
    else:
        row.extend([0.0] * 63)
    
    # Pose Arm Landmarks (3 points: 12-Shoulder, 14-Elbow, 16-Wrist x 4 = 12 features)
    if results.pose_landmarks:
        landmarks = results.pose_landmarks.landmark
        for idx in [12, 14, 16]:
            lm = landmarks[idx]
            row.extend([lm.x, lm.y, lm.z, lm.visibility])
    else:
        row.extend([0.0] * 12)
    return np.array(row, dtype=np.float32)

In [ ]:
# @title 3. Preprocessing: Selective Normalization & Sequence Padding
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# Dataset Paths on Google Drive
DATASET_PATH = '/content/drive/MyDrive/dataset_hijaiyah'

# Load tensors or demonstrate pipeline
if os.path.exists(os.path.join(DATASET_PATH, 'X_padded.npy')):
    X = np.load(os.path.join(DATASET_PATH, 'X_padded.npy'))
    y = np.load(os.path.join(DATASET_PATH, 'y_encoded.npy'))
    print(f"Loaded dataset: X={X.shape}, y={y.shape}")
else:
    print("Demo Mode: Generating synthetic tensor (112 classes, (20, 75))...")
    num_samples = 112 * 10
    X = np.random.uniform(0.0, 1.0, (num_samples, 20, 75)).astype(np.float32)
    y_indices = np.repeat(np.arange(112), 10)
    y = tf.keras.utils.to_categorical(y_indices, num_classes=112)

# Stratified Train-Test Split (70:30 SOTA Scenario)
y_stratify = np.argmax(y, axis=1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y_stratify
)
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

In [ ]:
# @title 4. Deep Learning Architectures Definition
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import (
    Input, MultiHeadAttention, LayerNormalization, Conv1D,
    GlobalAveragePooling1D, Dropout, Dense, Bidirectional, LSTM, GRU
)
from tensorflow.keras.optimizers import Adam

def create_transformer_model(input_shape=(20, 75), num_classes=112):
    inputs = Input(shape=input_shape)
    attn = MultiHeadAttention(num_heads=4, key_dim=64)(inputs, inputs)
    attn = LayerNormalization(epsilon=1e-6)(inputs + attn)
    ffn = Conv1D(filters=64, kernel_size=1, activation='relu')(attn)
    x = LayerNormalization(epsilon=1e-6)(attn + ffn)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=inputs, outputs=outputs, name='Transformer_Encoder')
    model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['categorical_accuracy'])
    return model

def create_bilstm_model(input_shape=(20, 75), num_classes=112):
    model = Sequential([
        Bidirectional(LSTM(64, return_sequences=False), input_shape=input_shape),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dense(num_classes, activation='softmax')
    ], name='Bi_LSTM_Baseline')
    model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['categorical_accuracy'])
    return model

def create_hybrid_model(input_shape=(20, 75), num_classes=112):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=input_shape),
        GRU(64, return_sequences=False),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ], name='Hybrid_Stacked_Baseline')
    model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['categorical_accuracy'])
    return model

print("All 3 model architectures compiled successfully!")

In [ ]:
# @title 5. Training Transformer Encoder with Callbacks
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

model = create_transformer_model(input_shape=(20, 75), num_classes=112)
callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    ModelCheckpoint('transformer_best.h5', monitor='val_categorical_accuracy', save_best_only=True, mode='max', verbose=1)
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=200,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# @title 6. Quantitative Evaluation (Weighted Metrics & Confusion Matrix)
from sklearn.metrics import classification_report, confusion_matrix

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)
print("=" * 40)
print(f"Accuracy : {report.get('accuracy', 0) * 100:.2f}%")
print(f"Precision: {report['weighted avg']['precision'] * 100:.2f}%")
print(f"Recall   : {report['weighted avg']['recall'] * 100:.2f}%")
print(f"F1-Score : {report['weighted avg']['f1-score'] * 100:.2f}%")
print("=" * 40)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap='Blues')
plt.title('Confusion Matrix - Transformer Encoder (112 Classes)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.show()